In [ ]:
import pandas as pd
import numpy as np


In [4]:
df = pd.read_csv('leakage_data_export.csv')
df['leakage_date'] = pd.to_datetime(df['leakage_date'], dayfirst=True)

print(f"Rows: {len(df)}")
print(f"Total Leakage: ${df['leakage_amount'].sum():,.2f}")
display(df.head())

Rows: 549
Total Leakage: $18,348.15


,leakage_date,leakage_category,leakage_amount,plan_name,region
0,2024-01-01,Partial Payment,0.96,Basic,East
1,2024-01-01,Unpaid Invoice,50.00,Pro,North
2,2024-01-01,Unpaid Invoice,10.00,Basic,East
3,2024-01-01,Partial Payment,6.91,Enterprise,West
4,2024-01-01,Partial Payment,14.73,Enterprise,South


In [5]:
# 1: Distribution Shape 
mean_loss = df['leakage_amount'].mean()
median_loss = df['leakage_amount'].median()
print(f"Mean Loss:   ${mean_loss:,.2f}")
print(f"Median Loss: ${median_loss:,.2f}")

Mean Loss:   $33.42
Median Loss: $9.05


In [6]:
# 2: Pareto Risk by Plan (80/20 Rule)
plan_pareto = (
    df.groupby('plan_name')['leakage_amount']
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)
plan_pareto['cumulative_pct'] = (
    plan_pareto['leakage_amount'].cumsum() /
    plan_pareto['leakage_amount'].sum() * 100
)
display(plan_pareto)

,plan_name,leakage_amount,cumulative_pct
0,Enterprise,12994.56,70.822181
1,Pro,4514.64,95.427604
2,Basic,838.95,100.000000


In [7]:
# 3: The "Kill Zone" (Region × Plan)
kill_zone = pd.pivot_table(
    df,
    values='leakage_amount',
    index='region',
    columns='plan_name',
    aggfunc='sum',
    fill_value=0
)

display(kill_zone.style.background_gradient(cmap='Reds').format("${:,.0f}"))

plan_name,Basic,Enterprise,Pro
region,,,
East,$205,"$3,855",$968
North,$265,"$3,077","$1,241"
South,$170,"$3,206",$868
West,$198,"$2,856","$1,437"
